In [1]:
import json
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors, FilterCatalog

rules = json.loads(Path("../models/rules.json").read_text())

rosemary = (
    pd.read_csv("../data/cultivar_compound_catalogue.csv")
    .drop_duplicates("compound_id")
)

# PAINS via RDKit catalogs (source is locked in rules.json by catalog name,
# not by inlined SMARTS; see rules.json["pains_catalog"]).
params = FilterCatalog.FilterCatalogParams()
for cat_name in rules["pains_catalog"]["catalogs"]:
    params.AddCatalog(getattr(FilterCatalog.FilterCatalogParams.FilterCatalogs, cat_name))
pains_catalog = FilterCatalog.FilterCatalog(params)

# Aggregator SMARTS from rules.json — compiled once, applied per compound.
agg_smarts_compiled = [
    (r["id"], Chem.MolFromSmarts(r["smarts"]))
    for r in rules["aggregator_rules"] if r["type"] == "smarts"
]

def is_aggregator_property(mol, mw, logp):
    """AGG-02: amphiphilic large-molecule check."""
    for r in rules["aggregator_rules"]:
        if r["id"] != "AGG-02":
            continue
        conds = r["conditions"]
        if not (logp > conds["MolLogP"]["value"] and mw > conds["MolWt"]["value"]):
            return False
    return True

dev_results = []
for _, row in rosemary.iterrows():
    mol = Chem.MolFromSmiles(row["canonical_smiles"])
    if mol is None:
        raise RuntimeError(
            f"Cannot parse SMILES for {row['compound_id']!r}: "
            f"{row['canonical_smiles']!r}. Fix the input rather than silently skipping."
        )

    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd  = Lipinski.NumHDonors(mol)
    hba  = Lipinski.NumHAcceptors(mol)
    rotb = rdMolDescriptors.CalcNumRotatableBonds(mol)
    psa  = rdMolDescriptors.CalcTPSA(mol)

    lipinski = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)
    veber    = (rotb <= 10) and (psa <= 140)
    pains    = not pains_catalog.HasMatch(mol)  # True = passes (not a PAINS)

    smarts_hits = [rid for rid, patt in agg_smarts_compiled
                   if patt is not None and mol.HasSubstructMatch(patt)]
    prop_hit    = is_aggregator_property(mol, mw, logp)
    aggregator_flag = bool(smarts_hits) or prop_hit   # True = IS an aggregator
    agg = not aggregator_flag                         # True = passes aggregator check

    dev_score = sum([lipinski, veber, pains, agg]) / 4.0

    dev_results.append({
        "compound_id": row["compound_id"],
        "developability_score": dev_score,
        "lipinski": lipinski,
        "veber": veber,
        "pains": pains,
        "aggregator": agg,
        "aggregator_flag": aggregator_flag,
        "aggregator_smarts_hits": ";".join(smarts_hits) if smarts_hits else "",
    })

pd.DataFrame(dev_results).to_csv("../results/developability_scores.csv", index=False)
print(f"Developability scores saved for {len(dev_results)} compounds.")

Developability scores saved for 28 compounds.
